# 10. Térbeli Autokorreláció (Moran's I) és Hotspot Elemzés

**Cél**: A kőbányai ingatlanárak térbeli klasztereződésének (Spatial Autocorrelation), forrópontjainak (Hotspots) és hidegpontjainak (Coldspots) ökonometriai kimutatása.

**Módszertan**:
- **Globális Moran's I**: Annak vizsgálata, hogy az ingatlanárak véletlenszerűen oszlanak-e el a térben, vagy statisztikailag szignifikáns térbeli klasztereződést mutatnak ($I > 0$).
- **Lokális Moran (LISA - Local Indicators of Spatial Association)**: A térbeli alcsoportok azonosítása: High-High (magas árak magas árú szomszédokkal), Low-Low (alacsony árak alacsony árú szomszédokkal), High-Low és Low-High térbeli kiugró értékek (outliers).

**Adatalap és mintaméret**: A teljes pontos GIS adatbázis N = 296 db (254 eladó + 42 kiadó). Az autokorrelációs vizsgálat a garantált pontos eladó lakások almintáján (N = 254 db) fut az eladási árak homogén térbeli struktúrájának kimutatására.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

df = load_szamitott_master()
df_geo = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].dropna(subset=['geokodolt_lat', 'geokodolt_lon', 'nm_ar_huf']).copy()
df_geo.reset_index(drop=True, inplace=True)
coords = np.column_stack((df_geo['geokodolt_lon'], df_geo['geokodolt_lat']))

print(f"Elemzett garantált pontos eladó lakásminta: {len(df_geo)} db.")

Elemzett garantált pontos eladó lakásminta: 254 db.


### 1. Globális Moran's I Térökonometriai Eredmények
A k=8 legközelebbi szomszéd (KNN) térbeli súlyozási mátrix és a 999 permutációs szimuláció alapján becsült autokorreláció.

In [2]:
# KNN súlymátrix és Globális Moran's I becslés
w_knn = KNN.from_array(coords, k=8)
w_knn.transform = 'R'

y_val = df_geo['nm_ar_huf'].values
moran_global = Moran(y_val, w_knn, permutations=999)

kpi_cards = [
    ("Globális Moran's I", f"{moran_global.I:.3f}", "Térbeli autokorreláció", "#1e3a8a"),
    ("Várható I (Véletlen)", f"{moran_global.EI:.3f}", "H0 hipotézis értéke", "#64748b"),
    ("Z-statisztika", f"{moran_global.z_sim:.2f}", "Szignifikancia mértéke", "#059669"),
    ("P-érték (p < 0.001)", f"{moran_global.p_sim:.4f}", "Statisztikailag szignifikáns", "#2563eb"),
    ("Térbeli Mintázat", "Erős Pozitív Klaszter", "Hasonló árak együtt", "#7c3aed"),
    ("Szomszédok száma (k)", "k = 8", "KNN topológia", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Moran Scatter Plot és Permutációs Hipotézisvizsgálat
A standardizált négyzetméterárak és a térbeli késleltetett értékek (Spatial Lag) összefüggése a 4 kvadránssal (HH, HL, LH, LL).

In [3]:
# Standardizálás és térbeli lag
z = (y_val - y_val.mean()) / y_val.std()
lag_z = w_knn.sparse.dot(z)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=z, y=lag_z, mode='markers',
    marker=dict(color='#2563eb', size=8, opacity=0.7),
    name='Ingatlanok'
))

# Regressziós egyenes (Moran's I a meredekség)
x_line = np.linspace(z.min(), z.max(), 100)
fig1.add_trace(go.Scatter(
    x=x_line, y=moran_global.I * x_line, mode='lines',
    line=dict(color='red', width=2),
    name=f"Moran's I lejtés ({moran_global.I:.3f})"
))

fig1.add_hline(y=0, line_dash='dash', line_color='gray')
fig1.add_vline(x=0, line_dash='dash', line_color='gray')

fig1.update_layout(
    title="Moran Pontdiagram (Moran Scatter Plot) - Négyzetméterár",
    xaxis_title='Standardizált Ár / m² (z)',
    yaxis_title='Térbeli Lag (Spatial Lag Wz)',
    template=PLOTLY_TEMPLATE,
    height=450
)
fig1.show()

# Permutációs eloszlás hisztogram
fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=moran_global.sim, nbinsx=35,
    name='Véletlen szimulált I értékek (999 db)',
    marker_color='#94a3b8', opacity=0.75
))
fig2.add_vline(
    x=moran_global.I, line_color='red', line_width=3,
    annotation_text=f'Megfigyelt Moran I: {moran_global.I:.3f} (p={moran_global.p_sim:.4f})'
)
fig2.update_layout(
    title='Globális Moran I Permutációs Referencia Eloszlás',
    xaxis_title='Szimulált I érték',
    yaxis_title='Gyakoriság',
    template=PLOTLY_TEMPLATE,
    height=380
)
fig2.show()

### 3. Lokális Térbeli Klaszterek (LISA Hotspot és Coldspot Térkép)
A statisztikailag szignifikáns High-High (Hotspot, magas árak) és Low-Low (Coldspot, alacsony árak) területek valós térképi megjelenítése.

In [4]:
# Lokális Moran LISA számítás
lisa = Moran_Local(y_val, w_knn, permutations=999)

# Kategóriák képzése
# q: 1=HH, 2=LH, 3=LL, 4=HL
labels_lisa = []
colors_lisa = []
for i in range(len(df_geo)):
    if lisa.p_sim[i] > 0.05:
        labels_lisa.append('Nem szignifikáns')
        colors_lisa.append('#94a3b8')
    elif lisa.q[i] == 1:
        labels_lisa.append('High-High (Hotspot)')
        colors_lisa.append('#ef4444')
    elif lisa.q[i] == 3:
        labels_lisa.append('Low-Low (Coldspot)')
        colors_lisa.append('#3b82f6')
    elif lisa.q[i] == 2:
        labels_lisa.append('Low-High (Környezet magasabb)')
        colors_lisa.append('#06b6d4')
    else:
        labels_lisa.append('High-Low (Környezet alacsonyabb)')
        colors_lisa.append('#f59e0b')

df_geo['LISA_Tipus'] = labels_lisa

fig3 = px.scatter_map(
    df_geo,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    color='LISA_Tipus',
    size=[10 if t != 'Nem szignifikáns' else 5 for t in labels_lisa],
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'varosresz': True, 'LISA_Tipus': True},
    color_discrete_map={
        'High-High (Hotspot)': '#ef4444',
        'Low-Low (Coldspot)': '#3b82f6',
        'Low-High (Környezet magasabb)': '#06b6d4',
        'High-Low (Környezet alacsonyabb)': '#f59e0b',
        'Nem szignifikáns': '#94a3b8'
    },
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title='LISA Térbeli Klaszterek és Kiugró Értékek Térképe (p < 0.05 szinten)'
)
fig3.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
fig3.show()

# Összefoglaló statisztika
lisa_stat = df_geo.groupby('LISA_Tipus').agg(
    Darab=('nm_ar_huf', 'count'),
    Median_Ar=('nm_ar_huf', 'median')
).reset_index()
lisa_stat['Median_Ar'] = lisa_stat['Median_Ar'].apply(fmt_huf)
display(HTML("<div style='max-width: 550px; margin: 15px 0;'>" + lisa_stat.to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

LISA_Tipus,Darab,Median_Ar
High-High (Hotspot),21,1 502 000 Ft
High-Low (Környezet alacsonyabb),11,1 436 000 Ft
Low-High (Környezet magasabb),3,985 135 Ft
Low-Low (Coldspot),65,1 098 000 Ft
Nem szignifikáns,154,1 432 224 Ft


### 4. Interaktív Moran's I és Súlyozási Pult
Tesztelje a térbeli autokorrelációt különböző változókra és szomszédsági k-értékekre!

In [5]:
w_var = widgets.Dropdown(
    options=[('Négyzetméterár', 'nm_ar_huf'), ('Alapterület', 'alapterulet_nm'), ('Állapot Kód', 'allapot_kod'), ('Szobaszám', 'szobaszam_osszes')],
    value='nm_ar_huf',
    description='Változó:'
)
w_k = widgets.IntSlider(min=4, max=16, step=2, value=8, description='k (KNN):')

out_moran = widgets.Output()

def frissit_moran(*args):
    v = w_var.value
    y = df_geo[v].values
    w = KNN.from_array(coords, k=w_k.value)
    w.transform = 'R'
    m = Moran(y, w, permutations=99)
    
    with out_moran:
        clear_output(wait=True)
        m_kpis = [
            ("Választott Moran I", f"{m.I:.3f}", f"z = {m.z_sim:.2f}", "#2563eb"),
            ("P-érték", f"{m.p_sim:.4f}", "Szignifikancia", "#059669" if m.p_sim < 0.05 else "#ef4444"),
            ("Szomszédok", f"k = {w_k.value}", "KNN mátrix", "#7c3aed")
        ]
        display(HTML(kpi_grid_html(m_kpis)))

w_var.observe(frissit_moran, names='value')
w_k.observe(frissit_moran, names='value')

display(widgets.HBox([w_var, w_k]))
display(out_moran)
frissit_moran()

Output()